# MERCon ARR Shape Ablation Notebook

Use this on the free Kaggle machine while Teacher-K ablation is running elsewhere.

Purpose: address the reviewer concern that ARR uses a simple linear timestep-dependent scaling without comparing alternatives. This notebook runs **inference-only** ablations on LOL-v2 Real using the current `best_distill_K5.pth` checkpoint.

It tests ARR factor shapes:

- `linear`: current paper, `1 - alpha * t_norm`
- `quadratic`: gentler early damping, `1 - alpha * t_norm^2`
- `sqrt`: stronger early damping, `1 - alpha * sqrt(t_norm)`
- `cosine`: smooth nonlinear schedule, `1 - alpha * (1 - cos(pi*t_norm/2))`

Decision rule: do not change the paper headline unless a LPIPS-enabled rerun clearly beats the current `23.86 PSNR / 0.807 SSIM / 0.200 LPIPS`. If no shape beats linear convincingly, use the result only to justify keeping the simple linear ARR.


## Required Kaggle Inputs

Attach these datasets before running:

1. `best_distill_K5.pth` checkpoint dataset.
2. LOL-v2 dataset, e.g. `tanhyml/lol-v2-dataset`, containing `LOL-v2/Real_captured/Test/Low` and `LOL-v2/Real_captured/Test/Normal`.

Use GPU. Internet ON.


In [ ]:
# Cell 1: Setup
!git clone https://github.com/chirana07/Diffusion_new_final.git /kaggle/working/Diffusion_new_final
!pip install -q lpips fvcore

from pathlib import Path
import os, glob, shutil, subprocess, sys, math
import pandas as pd

SRC = Path('/kaggle/working/Diffusion_new_final')
CODE = Path('/kaggle/working/src_arr_shape')
if CODE.exists():
    shutil.rmtree(CODE)
shutil.copytree(SRC, CODE)

OUT = Path('/kaggle/working/arr_shape_outputs')
OUT.mkdir(parents=True, exist_ok=True)

print('Code:', CODE)
print('Output:', OUT)


In [ ]:
# Cell 2: Locate checkpoint and LOL-v2 Real test split
from pathlib import Path
import glob, os

student_candidates = sorted(glob.glob('/kaggle/input/**/best_distill_K5.pth', recursive=True))
print('best_distill_K5 candidates:')
for c in student_candidates:
    print(c, os.path.getsize(c) / 1e6, 'MB')
if not student_candidates:
    raise FileNotFoundError('best_distill_K5.pth not found. Attach the distilled checkpoint dataset.')
BEST_DISTILL_CKPT = student_candidates[0]

DATASET_ROOT = Path('/kaggle/input/datasets/tanhyml/lol-v2-dataset/LOL-v2')
if not DATASET_ROOT.exists():
    hits = [p for p in Path('/kaggle/input').rglob('LOL-v2') if (p / 'Real_captured').is_dir()]
    if not hits:
        raise FileNotFoundError('Could not find LOL-v2/Real_captured under /kaggle/input. Attach tanhyml/lol-v2-dataset or equivalent.')
    DATASET_ROOT = hits[0]

LOLV2_LOW = DATASET_ROOT / 'Real_captured/Test/Low'
LOLV2_HIGH = DATASET_ROOT / 'Real_captured/Test/Normal'

for p in [LOLV2_LOW, LOLV2_HIGH]:
    print(p, p.is_dir(), len(list(p.glob('*'))) if p.is_dir() else 0)

print('BEST_DISTILL_CKPT =', BEST_DISTILL_CKPT)
print('DATASET_ROOT =', DATASET_ROOT)


In [ ]:
# Cell 3: Patch evaluation.py and diffusion.py to support ARR shape
from pathlib import Path

diff_path = CODE / 'diffusion.py'
eval_path = CODE / 'evaluation.py'

diff_txt = diff_path.read_text()

# Add gate_shape parameter to function signature.
diff_txt = diff_txt.replace(
    'def ddim_sample(self, model, low_light, inference_steps=None, eta=0.0,\n                    gate_alpha=0.0, gate_floor=0.5):',
    'def ddim_sample(self, model, low_light, inference_steps=None, eta=0.0,\n                    gate_alpha=0.0, gate_floor=0.5, gate_shape="linear"):'
)

old_block = '''            # --- Adaptive Residual Rescaling (ARR) ---
            if gate_alpha > 0.0:
                # t is identical across the batch in DDIM; safe to take a scalar.
                t_norm = float(int(i)) / T_minus_1
                factor = max(float(gate_floor), 1.0 - gate_alpha * t_norm)
                x0_pred = x0_pred * factor
'''

new_block = '''            # --- Adaptive Residual Rescaling (ARR) ---
            if gate_alpha > 0.0:
                # t is identical across the batch in DDIM; safe to take a scalar.
                t_norm = float(int(i)) / T_minus_1
                if gate_shape == "linear":
                    shape_term = t_norm
                elif gate_shape == "quadratic":
                    shape_term = t_norm ** 2
                elif gate_shape == "sqrt":
                    shape_term = math.sqrt(max(t_norm, 0.0))
                elif gate_shape == "cosine":
                    shape_term = 1.0 - math.cos(0.5 * math.pi * t_norm)
                else:
                    raise ValueError(f"Unknown gate_shape: {gate_shape}")
                factor = max(float(gate_floor), 1.0 - gate_alpha * shape_term)
                x0_pred = x0_pred * factor
'''

if old_block not in diff_txt:
    raise RuntimeError('Could not find ARR block in diffusion.py; repository changed.')
diff_txt = diff_txt.replace(old_block, new_block)
diff_path.write_text(diff_txt)

eval_txt = eval_path.read_text()

# evaluate_split signature
eval_txt = eval_txt.replace('gate_alpha=0.0,\n    gate_floor=0.5,', 'gate_alpha=0.0,\n    gate_floor=0.5,\n    gate_shape="linear",')

# ddim_sample call
eval_txt = eval_txt.replace(
    'gate_alpha=gate_alpha, gate_floor=gate_floor,',
    'gate_alpha=gate_alpha, gate_floor=gate_floor, gate_shape=gate_shape,'
)

# CLI arg
eval_txt = eval_txt.replace(
    'parser.add_argument(\n        "--gate-floor", type=float, default=0.5,\n        help="Floor on the ARR factor (prevents zeroing the residual at noisy steps).",\n    )',
    'parser.add_argument(\n        "--gate-floor", type=float, default=0.5,\n        help="Floor on the ARR factor (prevents zeroing the residual at noisy steps).",\n    )\n    parser.add_argument(\n        "--gate-shape", choices=["linear", "quadratic", "sqrt", "cosine"], default="linear",\n        help="ARR timestep shape. linear is the paper default."\n    )'
)

# evaluate_split call in main
eval_txt = eval_txt.replace(
    'gate_alpha=args.gate_alpha,\n            gate_floor=args.gate_floor,',
    'gate_alpha=args.gate_alpha,\n            gate_floor=args.gate_floor,\n            gate_shape=args.gate_shape,'
)

eval_path.write_text(eval_txt)

print('Patched diffusion.py and evaluation.py')
!grep -n "gate_shape" /kaggle/working/src_arr_shape/diffusion.py /kaggle/working/src_arr_shape/evaluation.py


In [ ]:
# Cell 4: Evaluation helper
import subprocess, sys
from pathlib import Path
import pandas as pd

def run_eval(shape, alpha, floor=0.5, steps=5, lpips=False):
    tag = f'shape_{shape}_a{int(alpha*100):03d}_f{int(floor*100):03d}'
    root = OUT / 'eval'
    cmd = [
        sys.executable, 'evaluation.py',
        '--splits', f'lolv2_real:{LOLV2_LOW}:{LOLV2_HIGH}',
        '--checkpoint', str(BEST_DISTILL_CKPT),
        '--inference-steps', str(steps),
        '--sampler', 'ddim',
        '--gate-alpha', str(alpha),
        '--gate-floor', str(floor),
        '--gate-shape', shape,
        '--results-root', str(root),
        '--tag', tag,
    ]
    if not lpips:
        cmd.append('--no-lpips')
    print('\nRunning:', ' '.join(cmd))
    subprocess.run(cmd, cwd=CODE, check=True)
    summary = root.parent / f'eval_{tag}' / 'summary.csv'
    df = pd.read_csv(summary)
    display(df)
    row = df.iloc[0].to_dict()
    row['shape'] = shape
    row['alpha'] = alpha
    row['floor'] = floor
    row['steps'] = steps
    row['lpips_eval'] = lpips
    row['summary_path'] = str(summary)
    return summary, row


In [ ]:
# Cell 5: ARR shape sweep, no LPIPS for speed
# This is the main ablation. Expected runtime: roughly 10-20 minutes on T4.
shapes = ['linear', 'quadratic', 'sqrt', 'cosine']
alphas = [0.3, 0.4, 0.5, 0.6]
floor = 0.5

rows = []
for shape in shapes:
    for alpha in alphas:
        summary, row = run_eval(shape=shape, alpha=alpha, floor=floor, steps=5, lpips=False)
        rows.append(row)
        sweep_df = pd.DataFrame(rows).sort_values(['psnr_mean', 'ssim_mean'], ascending=False)
        sweep_df.to_csv('/kaggle/working/arr_shape_sweep.csv', index=False)
        display(sweep_df[['shape','alpha','floor','n','psnr_mean','ssim_mean','runtime_mean','summary_path']])
        print('Saved partial: /kaggle/working/arr_shape_sweep.csv')

sweep_df = pd.DataFrame(rows).sort_values(['psnr_mean', 'ssim_mean'], ascending=False)
display(sweep_df[['shape','alpha','floor','n','psnr_mean','ssim_mean','psnr_std','ssim_std','runtime_mean','summary_path']])
sweep_df.to_csv('/kaggle/working/arr_shape_sweep.csv', index=False)
print('Saved final: /kaggle/working/arr_shape_sweep.csv')


In [ ]:
# Cell 6: Final LPIPS checks for paper-default linear and best non-linear candidate
# Run this after Cell 5. It does LPIPS only for a small number of important rows.
sweep_df = pd.read_csv('/kaggle/working/arr_shape_sweep.csv')
ranked = sweep_df.sort_values(['psnr_mean', 'ssim_mean'], ascending=False).reset_index(drop=True)
display(ranked[['shape','alpha','floor','psnr_mean','ssim_mean','runtime_mean']].head(8))

checks = []
# Always check the current paper default.
checks.append(('linear', 0.5, 0.5))

# Check the best overall row.
best = ranked.iloc[0]
checks.append((str(best['shape']), float(best['alpha']), float(best['floor'])))

# Check the best non-linear row if different.
nonlinear = ranked[ranked['shape'] != 'linear']
if len(nonlinear):
    best_nonlin = nonlinear.iloc[0]
    checks.append((str(best_nonlin['shape']), float(best_nonlin['alpha']), float(best_nonlin['floor'])))

# Deduplicate while preserving order.
seen = set()
unique_checks = []
for item in checks:
    key = (item[0], round(item[1], 6), round(item[2], 6))
    if key not in seen:
        seen.add(key)
        unique_checks.append(item)

rows = []
for shape, alpha, floor in unique_checks:
    summary, row = run_eval(shape=shape, alpha=alpha, floor=floor, steps=5, lpips=True)
    rows.append(row)

lpips_df = pd.DataFrame(rows).sort_values(['psnr_mean', 'ssim_mean'], ascending=False)
display(lpips_df[['shape','alpha','floor','n','psnr_mean','ssim_mean','lpips_mean','runtime_mean','summary_path']])
lpips_df.to_csv('/kaggle/working/arr_shape_lpips_checks.csv', index=False)
print('Saved: /kaggle/working/arr_shape_lpips_checks.csv')


In [ ]:
# Cell 7: Export small CSVs only
!mkdir -p /kaggle/working/arr_shape_export
!cp /kaggle/working/arr_shape_sweep.csv /kaggle/working/arr_shape_export/ 2>/dev/null || true
!cp /kaggle/working/arr_shape_lpips_checks.csv /kaggle/working/arr_shape_export/ 2>/dev/null || true
!cd /kaggle/working && zip -r arr_shape_export.zip arr_shape_export >/dev/null
print('Download: /kaggle/working/arr_shape_export.zip')


## How to Use the Result

If `linear, alpha=0.5, floor=0.5` remains best or close to best, we can add one sentence to the paper/rebuttal style text:

> We also screened quadratic, square-root, and cosine ARR schedules on LOL-v2 Real; none gave a clear improvement over the linear schedule, so we retain the simpler one-parameter linear form.

If another shape clearly wins **and** its LPIPS-enabled result beats the paper headline, we can discuss whether to update the headline. Do not update the paper from the no-LPIPS sweep alone.
